In [3]:
# Setup and package imports 
import mlflow
from pyspark.ml import Pipeline
from pyspark.ml.feature import (
    StringIndexer, OneHotEncoder,
    VectorAssembler, MinMaxScaler
)

from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.classification import GBTClassifier

from synapse.ml.lightgbm import LightGBMClassifier


StatementMeta(, b41e1acd-e882-4c71-9c1a-4dd710391353, 4, Finished, Available, Finished, False)

In [5]:

# -------------------------------------------
# a. Read Dataset and create Spark DataFrame
# -------------------------------------------

spark_df = spark.read.table('FraudDetection.dbo.silver_data_for_mlmodel') #.select(features)
display( spark_df.limit(5) )

StatementMeta(, b41e1acd-e882-4c71-9c1a-4dd710391353, 6, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, dfa70674-1ff7-4da3-a5ed-0d019bb265d9)

In [6]:
# ** 4. Train / Test Split **
train_df, test_df = spark_df.randomSplit([0.8, 0.2], seed=42)

train_df.cache()
test_df.cache()

StatementMeta(, b41e1acd-e882-4c71-9c1a-4dd710391353, 7, Finished, Available, Finished, False)

DataFrame[step: bigint, type: string, amount: double, oldbalanceOrg: double, newbalanceOrig: double, oldbalanceDest: double, newbalanceDest: double, isFraud: bigint, orig_balance_diff: double, dest_balance_diff: double, orig_zero: int, dest_zero: int, log_amount: double, amount_to_balance: double, orig_error: int, dest_error: int, orig_name_type: string, dest_name_type: string, orig_name_freq: bigint, dest_name_freq: bigint, pair_freq: bigint, balance_ratio_orig: double, balance_ratio_dest: double, abs_orig_diff: double, abs_dest_diff: double, amount_vs_orig: double, amount_vs_dest: double, orig_zero_flag: int, dest_zero_flag: int, hour: bigint, is_night: int, sqrt_amount: double, log_balance_orig: double, log_balance_dest: double]

In [9]:
%run ././ml_common


StatementMeta(, b41e1acd-e882-4c71-9c1a-4dd710391353, 15, Finished, Available, Finished, True)

In [10]:
# -----------------------------------------
# 0. Get categorical and numerical features
# --------------------------------------------
label_col = "isFraud" 
categorical_cols, numeric_cols = auto_detect_colum_type(spark_df, label_col)

# **1. Preprocessing the pipeline** 
# ---------------------------------------
# 1.1 Encode categorical freatures 
# --------------------------------------
indexers, encoders = encode_categorical_features(categorical_cols)

# -------------------------------------------------------
# 1.2 Assemble features 
# ----------------------------
assembler_inputs = [f"{c}_ohe" for c in categorical_cols] + numeric_cols
print('Assembler Inputs: ', assembler_inputs)

assembler = VectorAssembler(
    inputCols=assembler_inputs,
    outputCol="features_raw"
)

# ----------------------------
# 1.3 Scaling: MimMax scaling 
# ----------------------------
scaler = MinMaxScaler(
    inputCol="features_raw",
    outputCol="features"
)


lgbm = GBTClassifier(
    featuresCol="features",
    labelCol=label_col
    #isUnbalance=True
)


# **3. Define MLflow pipeline **
pipeline = Pipeline(
    stages=[
        *indexers,
        *encoders,
        assembler,
        scaler,
        lgbm
    ]
)

# ** 4. Hyperparamer tunning: for  **
paramGrid = hyperparameter_tunning(lgbm, 'GBT')

# ** 5. Evaluate and validate the model **
evaluator, cv = evaluate_and_validate(label_col, pipeline, paramGrid)

StatementMeta(, b41e1acd-e882-4c71-9c1a-4dd710391353, 16, Finished, Available, Finished, False)

Categorical: ['type', 'orig_name_type', 'dest_name_type']
Numeric: ['step', 'amount', 'oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest', 'newbalanceDest', 'orig_balance_diff', 'dest_balance_diff', 'orig_zero', 'dest_zero', 'log_amount', 'amount_to_balance', 'orig_error', 'dest_error', 'orig_name_freq', 'dest_name_freq', 'pair_freq', 'balance_ratio_orig', 'balance_ratio_dest', 'abs_orig_diff', 'abs_dest_diff', 'amount_vs_orig', 'amount_vs_dest', 'orig_zero_flag', 'dest_zero_flag', 'hour', 'is_night', 'sqrt_amount', 'log_balance_orig', 'log_balance_dest']
Assembler Inputs:  ['type_ohe', 'orig_name_type_ohe', 'dest_name_type_ohe', 'step', 'amount', 'oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest', 'newbalanceDest', 'orig_balance_diff', 'dest_balance_diff', 'orig_zero', 'dest_zero', 'log_amount', 'amount_to_balance', 'orig_error', 'dest_error', 'orig_name_freq', 'dest_name_freq', 'pair_freq', 'balance_ratio_orig', 'balance_ratio_dest', 'abs_orig_diff', 'abs_dest_diff', 'amount_vs_orig

In [1]:
# ---------------------------------------------
# 4. MLflow tracking (Fabric auto-integrated) -
# ---------------------------------------------
mlflow.set_experiment("fabric_spark_GBTClassifier")

with mlflow.start_run():

    model = cv.fit(train_df)

    best_model = model.bestModel

    # Log best params manually
    best_lr = best_model.stages[-1]
    

    mlflow.log_param("maxDepth", best_lr.getMaxDepth())
    mlflow.log_param("stepSize", best_lr.getStepSize())
    mlflow.log_param("maxIter", best_lr.getMaxIter())

    # Evaluate
    preds = best_model.transform(test_df)
    auc = evaluator.evaluate(preds)

    mlflow.log_metric("AUC", auc)
    mlflow.spark.log_model(best_model, "model")



    print("Best AUC:", auc)

StatementMeta(, 64d4793a-f671-44c8-8d5d-d8c427cc09d5, 3, Finished, Available, Finished, False)

NameError: name 'mlflow' is not defined

In [ ]:
mlflow.register_model(
    f"runs:/807afb0d-578d-4f44-a8ad-09bdf6f87f2e/model",
    "GBTClassifier_FraudDetection_Model"
)

StatementMeta(, cfe1cf35-5491-474e-9261-f168b6e7c1eb, 9, Finished, Available, Finished, False)

Successfully registered model 'GBTClassifier_FraudDetection_Model'.


2026/06/04 09:57:22 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: GBTClassifier_FraudDetection_Model, version 1
Created version '1' of model 'GBTClassifier_FraudDetection_Model'.


<ModelVersion: aliases=[], creation_timestamp=1780567042923, current_stage='None', description='', last_updated_timestamp=1780567042923, name='GBTClassifier_FraudDetection_Model', run_id='807afb0d-578d-4f44-a8ad-09bdf6f87f2e', run_link='', source='abfss://f8e3d19d-f7c2-4966-9804-d462790c6dac@northeurope-onelake.dfs.core.windows.net/7e9638a8-12c5-4dda-8386-755064d0dc2f/Data/33bf2fd1-900a-4f10-a4d7-82d2ff19cd16/artifacts', status='READY', status_message='', tags={'synapseml.user.id': 'cf56f162-c721-4ffc-93ea-5618392cedee',
 'synapseml.user.name': 'Kassa, Abay Molla'}, user_id='cf56f162-c721-4ffc-93ea-5618392cedee', version='1'>

In [ ]:
mlflow.end_run()

StatementMeta(, a5f99239-1c2d-41a1-a6ef-f52ee1740f7f, -1, Cancelled, , Cancelled, True)

In [ ]:
# ROC curve plot

from sklearn.metrics import roc_curve, auc
import matplotlib.pyplot as plt

# Extract probabilities
preds_pd = preds.toPandas()[["isFraud", "probability"]]
preds_pd["prob"] = preds_pd["probability"].apply(lambda x: x[1])

fpr, tpr, thresholds = roc_curve(preds_pd["isFraud"], preds_pd["prob"])
roc_auc = auc(fpr, tpr)

plt.figure()
plt.plot(fpr, tpr, label=f"ROC curve (AUC = {roc_auc:.3f})")
plt.plot([0, 1], [0, 1], linestyle="--")

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()

plt.show()

StatementMeta(, cfe1cf35-5491-474e-9261-f168b6e7c1eb, 3, Finished, Available, Finished, False)

NameError: name 'preds' is not defined

In [7]:
dbutils.fs.cp(
    "file:/tmp/tmpxsqfvvon/",
    "C:\Users\akassa\FABRIC",
    recurse=True
)



StatementMeta(, 68589c2f-0d3a-4446-8548-c1b1afeb2fd9, 11, Finished, Available, Finished, False)

SyntaxError: (unicode error) 'unicodeescape' codec can't decode bytes in position 2-3: truncated \UXXXXXXXX escape (2150285134.py, line 3)